In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('../DataSets_File/EDA_file/titanic_data_updated.csv')

In [3]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
484,485,yes,first,"Bishop, Mr. Dickinson H",male,25.0,1,0,11967,91.0792,B49,C
459,460,no,third,"O'Connor, Mr. Maurice",male,NaN,0,0,371060,7.7500,NaN,Q
799,800,no,third,"Van Impe, Mrs. Jean Baptiste (Rosalie Paula Go...",female,30.0,1,1,345773,24.1500,NaN,S
121,122,no,third,"Moore, Mr. Leonard Charles",male,NaN,0,0,A4. 54510,8.0500,NaN,S
543,544,yes,second,"Beane, Mr. Edward",male,32.0,1,0,2908,26.0000,NaN,S


In [4]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)

# Extract Feature Columns

In [5]:
X = df.drop(['Survived'],axis = 1) 
X

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,third,male,22.0,1,0,7.2500,NaN,S
1,first,female,38.0,1,0,71.2833,C85,C
2,third,female,26.0,0,0,7.9250,NaN,S
3,first,female,35.0,1,0,53.1000,C123,S
4,third,male,35.0,0,0,8.0500,NaN,S
...,...,...,...,...,...,...,...,...
886,second,male,27.0,0,0,13.0000,NaN,S
887,first,female,19.0,0,0,30.0000,B42,S
888,third,female,NaN,1,2,23.4500,NaN,S
889,first,male,26.0,0,0,30.0000,C148,C


# Extract Target Column

In [6]:
y = df['Survived']
y

0       no
1      yes
2      yes
3      yes
4       no
      ... 
886     no
887    yes
888     no
889    yes
890     no
Name: Survived, Length: 891, dtype: str

# Train Split Data

In [7]:
X_train,X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# First ColumnTransformer - Imputation

In [8]:
imputer_transformer = ColumnTransformer(
    transformers=[
        # Age column e mean imputation
        ('age',SimpleImputer(strategy='mean'),['Age']),

        # Embarked column - most frequent imputatin
        ('embarked',SimpleImputer(strategy= 'most_frequent'),['Embarked']),

        # Cabin column e constant imputation + missing indicator
        ('cabin',SimpleImputer(
            strategy='constant',
            fill_value='Missing',
            add_indicator= True
         ),['Cabin'])
    ],
    remainder= 'passthrough',
    verbose_feature_names_out= False
)

imputer_transformer.set_output(transform='pandas') #er karone numpy arry er poriborte pandas DataFrame hobe

#fit and transform

imputer_transformer.fit(X_train)
X_train = imputer_transformer.transform(X_train)
X_test = imputer_transformer.transform(X_test)

In [9]:
X_train

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,SibSp,Parch,Fare
331,45.500000,S,C124,False,first,male,0,0,28.5000
733,23.000000,S,Missing,True,second,male,0,0,13.0000
382,32.000000,S,Missing,True,third,male,0,0,7.9250
704,26.000000,S,Missing,True,third,male,1,0,7.8542
813,6.000000,S,Missing,True,third,female,4,2,31.2750
...,...,...,...,...,...,...,...,...,...
106,21.000000,S,Missing,True,third,female,0,0,7.6500
270,29.498846,S,Missing,True,first,male,0,0,31.0000
860,41.000000,S,Missing,True,third,male,2,0,14.1083
435,14.000000,S,B96 B98,False,first,female,1,2,120.0000


# Second ColumnTransformer - Encoding and Scaling

In [10]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler

encoder_scaler = ColumnTransformer(
    transformers=[
        #Pclass ordinal encoding
        ('pclass',OrdinalEncoder(categories=[['third','second','first']]),['Pclass']),

        # Embarked, Sex, Cabin_Deck - one hot encoder
        ('embarked',OneHotEncoder(sparse_output=False,drop='first',handle_unknown= 'ignore'),['Embarked','Sex','Cabin']),

        # Age - standard scaling
        ('age_scaler',StandardScaler(),['Age']),

        # Fare, FamilySize - minmax scaling
        ('fare_scaler',MinMaxScaler(),['Fare'])
    ],
    remainder= 'passthrough',
    verbose_feature_names_out= False
)

encoder_scaler.set_output(transform='pandas')

#fit and transfrom

encoder_scaler.fit(X_train)
X_train = encoder_scaler.transform(X_train)
X_test = encoder_scaler.transform(X_test)

e:\ML\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
